# 02 - Analise Textual Inicial

Objetivo: carregar a amostra processada das integras do STJ, gerar estatisticas descritivas simples e produzir um relatorio exploratorio em Markdown.

Este notebook ainda nao faz comparacao com LLM, classificacao, fine-tuning ou conclusoes sobre vies.

In [ ]:
from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# No Colab, descomente as duas linhas abaixo.
# from google.colab import drive
# drive.mount('/content/drive')

DRIVE_DATA = Path('/content/drive/MyDrive/Mestrado/2026/llms/data')
PROCESSED_DATA = DRIVE_DATA / 'processed'
REPORTS_DATA = DRIVE_DATA / 'reports' / 'summaries'
FIGURES_DATA = DRIVE_DATA / 'reports' / 'figures'

for path in [PROCESSED_DATA, REPORTS_DATA, FIGURES_DATA]:
    path.mkdir(parents=True, exist_ok=True)

sample_path = PROCESSED_DATA / 'stj_integras_sample.parquet'
sample_path

## 1. Carregar amostra

In [ ]:
if sample_path.exists():
    df = pd.read_parquet(sample_path)
else:
    df = pd.read_csv(PROCESSED_DATA / 'stj_integras_sample.csv')

print(f'Amostra: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
df.head()

In [ ]:
df.info()

## 2. Estatisticas de texto

In [ ]:
text_col = 'texto_limpo'

df[text_col] = df[text_col].fillna('').astype(str)
df['n_chars'] = df[text_col].str.len()
df['n_words'] = df[text_col].str.split().str.len()
df['texto_vazio'] = df[text_col].str.strip().eq('')

df[['n_chars', 'n_words']].describe()

In [ ]:
empty_share = df['texto_vazio'].mean() if len(df) else np.nan
print(f'Percentual de textos vazios: {empty_share:.2%}')

## 3. Distribuicoes dos metadados

In [ ]:
def value_counts_safe(df: pd.DataFrame, column: str, n: int = 20) -> pd.Series:
    if column not in df.columns:
        return pd.Series(dtype='int64', name=column)
    return df[column].fillna('(vazio)').astype(str).value_counts().head(n)

tipo_counts = value_counts_safe(df, 'tipoDocumento')
ministro_counts = value_counts_safe(df, 'ministro')
teor_counts = value_counts_safe(df, 'teor')

tipo_counts

In [ ]:
ministro_counts

In [ ]:
teor_counts

## 4. Graficos simples

In [ ]:
ax = df['n_words'].plot(kind='hist', bins=30, title='Distribuicao do tamanho dos textos')
ax.set_xlabel('Numero de palavras')
ax.set_ylabel('Documentos')
plt.tight_layout()
figure_path = FIGURES_DATA / 'stj_integras_tamanho_textos.png'
plt.savefig(figure_path, dpi=150)
figure_path

## 5. Termos frequentes preliminares

Esta contagem e apenas exploratoria. Nao aplicar stopwords, stemming ou lematizacao nesta fase sem decisao metodologica explicita.

In [ ]:
def tokenize_basic(text: str) -> list[str]:
    return re.findall(r'\b[\wÀ-ÿ]{3,}\b', text.lower())

counter = Counter()
for text in df[text_col]:
    counter.update(tokenize_basic(text))

common_terms = pd.DataFrame(counter.most_common(50), columns=['termo', 'frequencia'])
common_terms.head(20)

## 6. Exemplos para leitura qualitativa

In [ ]:
cols = [c for c in ['SeqDocumento', 'tipoDocumento', 'ministro', 'teor', 'processo', 'n_words'] if c in df.columns]
df.sort_values('n_words', ascending=False)[cols].head(10)

In [ ]:
example = df.sort_values('n_words', ascending=False).iloc[0]
print('SeqDocumento:', example.get('SeqDocumento'))
print('Tipo:', example.get('tipoDocumento'))
print('Ministro:', example.get('ministro'))
print('\nTrecho inicial:\n')
print(example[text_col][:3000])

## 7. Relatorio exploratorio

In [ ]:
def series_to_markdown(series: pd.Series) -> str:
    if series.empty:
        return '_Coluna ausente ou sem valores._'
    table = series.rename_axis('valor').reset_index(name='n')
    return table.to_markdown(index=False)

summary = df[['n_chars', 'n_words']].describe().round(2)
empty_pct = df['texto_vazio'].mean() * 100 if len(df) else 0

report = f'''# Relatorio exploratorio - STJ Integras

## Escopo

Analise inicial de amostra processada da base STJ - Integras de Decisoes Terminativas e Acordaos do Diario da Justica.

## Totais

- Total de documentos na amostra: {len(df):,}
- Percentual de textos vazios: {empty_pct:.2f}%
- Tamanho medio em caracteres: {df['n_chars'].mean():.2f}
- Tamanho minimo em caracteres: {df['n_chars'].min()}
- Tamanho maximo em caracteres: {df['n_chars'].max()}
- Tamanho medio em palavras: {df['n_words'].mean():.2f}

## Distribuicao por tipo de documento

{series_to_markdown(tipo_counts)}

## Distribuicao por ministro

{series_to_markdown(ministro_counts)}

## Distribuicao por teor

{series_to_markdown(teor_counts)}

## Estatisticas de tamanho

{summary.to_markdown()}

## Observacoes preliminares sobre qualidade da base

- Verificar se a amostra e representativa antes de extrapolar achados.
- Conferir manualmente documentos com texto vazio ou muito curto.
- Validar se `SeqDocumento` mantem alta cobertura entre metadados e TXT.
- Evitar conclusoes sobre vies ou comportamento de LLM nesta etapa.
'''

report_path = REPORTS_DATA / 'eda_summary.md'
report_path.write_text(report, encoding='utf-8')
report_path

## 8. Proximos passos

- Aumentar a amostra se a ligacao por `SeqDocumento` estiver consistente.
- Mover funcoes estaveis para `src/`.
- Definir criterios de recorte por tipo de documento, teor, ministro ou assunto.
- So depois avaliar embeddings, clustering ou comparacoes com LLM.